# Day 3 Exam: Data Quality & Infrastructure

**Notebooks covered:** NB03-06 (primary), NB00-02 (review ~20-40%)

---

**Rules:**
- Set a timer for each section (noted per problem)
- No documentation, no notebooks, no Google — closed book
- Honor system: if you can't recall something, leave it blank and note what you'd look up
- Write working code that passes the provided assertions
- If you finish early, refactor for clarity

**Time budget:**
| Section | Time |
|---|---|
| Warmup (3 problems) | 15 min total |
| Main Problem 1 | 30 min |
| Main Problem 2 | 30 min |
| Main Problem 3 | 30 min |
| **Total** | **~105 min** |

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from typing import Callable
import time

---

## Warmup (5 min each)

### Warmup 1: L2 Normalize Embeddings

Implement `l2_normalize(embeddings: torch.Tensor) -> torch.Tensor`

- Input: `(B, D)` tensor of embeddings
- Output: `(B, D)` tensor where each row has unit L2 norm
- Normalize along the last dimension
- Handle the zero-vector edge case (avoid division by zero)

In [ ]:
def l2_normalize(embeddings: torch.Tensor) -> torch.Tensor:
    """Normalize batch of embeddings to unit L2 norm along last dim."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Tests: Warmup 1 ---
torch.manual_seed(42)
emb = torch.randn(8, 128)
normed = l2_normalize(emb)

# Shape preserved
assert normed.shape == emb.shape, f"Shape mismatch: {normed.shape} vs {emb.shape}"

# Each row has unit norm
norms = torch.norm(normed, dim=-1)
assert torch.allclose(norms, torch.ones(8), atol=1e-5), f"Norms not unit: {norms}"

# Zero vector shouldn't produce NaN
zero_emb = torch.zeros(2, 64)
zero_normed = l2_normalize(zero_emb)
assert not torch.isnan(zero_normed).any(), "NaN from zero vector"

print("Warmup 1 PASSED")

---

### Warmup 2: Hamming Distance

Implement `hamming_distance(h1: int, h2: int) -> int`

- Count the number of differing bits between two integer hashes
- This is the core comparison metric for perceptual hashing (dHash, pHash)

In [ ]:
def hamming_distance(h1: int, h2: int) -> int:
    """Count differing bits between two integer hashes."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Tests: Warmup 2 ---
# Self-distance is always 0
assert hamming_distance(0, 0) == 0
assert hamming_distance(0xFF, 0xFF) == 0
assert hamming_distance(123456789, 123456789) == 0

# Known pairs
assert hamming_distance(0b1111, 0b0000) == 4, "All 4 bits differ"
assert hamming_distance(0b1010, 0b0101) == 4, "Alternating bits"
assert hamming_distance(0b1100, 0b1010) == 2, "Two bits differ"
assert hamming_distance(0, 1) == 1, "Single bit"

# Symmetry
assert hamming_distance(42, 97) == hamming_distance(97, 42)

print("Warmup 2 PASSED")

---

### Warmup 3: DataLoader Construction

Given the dataset below, create a `DataLoader` with:
- `batch_size=32`
- `shuffle=True`
- `pin_memory=True`
- `drop_last=True`

Then iterate one batch and print the shapes of each tensor in the batch.

In [ ]:
# Dataset is provided — create the DataLoader and iterate one batch
images = torch.randn(100, 3, 64, 64)
labels = torch.randint(0, 10, (100,))
dataset = TensorDataset(images, labels)

# YOUR CODE HERE: create loader, iterate one batch, print shapes
raise NotImplementedError

In [ ]:
# --- Tests: Warmup 3 ---
# `loader` must be defined in the cell above
batch = next(iter(loader))
batch_images, batch_labels = batch

assert batch_images.shape[0] == 32, f"Batch size should be 32, got {batch_images.shape[0]}"
assert batch_images.shape == (32, 3, 64, 64), f"Image shape wrong: {batch_images.shape}"
assert batch_labels.shape == (32,), f"Label shape wrong: {batch_labels.shape}"

# drop_last=True means we should have floor(100/32) = 3 batches (not 4)
n_batches = sum(1 for _ in loader)
assert n_batches == 3, f"Expected 3 batches with drop_last=True, got {n_batches}"

print("Warmup 3 PASSED")

---

## Main Problems (30 min each)

### Main Problem 1: Image Deduplication Pipeline

Implement an `ImageDeduplicator` class that uses difference hashing (dHash) to find and remove near-duplicate images.

**Class interface:**

```python
class ImageDeduplicator:
    def __init__(self, threshold: int = 5): ...
    def compute_hash(self, image: np.ndarray) -> int: ...
    def find_duplicates(self, images: list[np.ndarray]) -> list[set[int]]: ...
    def deduplicate(self, images: list[np.ndarray]) -> tuple[list[int], dict]: ...
```

**`compute_hash`:**
- Difference hash (dHash): resize image to 9x8 grayscale
- Compare each pixel to the one to its right (8 comparisons per row, 8 rows = 64 bits)
- Pack the boolean comparisons into a single integer
- Input is `(H, W)` or `(H, W, 3)` uint8 numpy array

**`find_duplicates`:**
- Compare all pairs of hashes
- Return clusters (list of sets) where each set contains indices of images whose pairwise Hamming distance is <= threshold
- Only return clusters with 2+ members
- Use union-find or simple clustering

**`deduplicate`:**
- Return `(kept_indices, report)` where:
  - `kept_indices`: sorted list of indices to keep (one per cluster, plus all singletons)
  - `report`: dict with keys `n_input`, `n_output`, `n_removed`, `clusters` (the list of sets)

In [ ]:
class ImageDeduplicator:
    """Deduplicate images using difference hashing."""

    def __init__(self, threshold: int = 5):
        # YOUR CODE HERE
        raise NotImplementedError

    def compute_hash(self, image: np.ndarray) -> int:
        """Difference hash: resize to 9x8 grayscale, compare adjacent pixels, pack bits."""
        # YOUR CODE HERE
        raise NotImplementedError

    def find_duplicates(self, images: list[np.ndarray]) -> list[set[int]]:
        """Return clusters of duplicate image indices (sets with 2+ members)."""
        # YOUR CODE HERE
        raise NotImplementedError

    def deduplicate(self, images: list[np.ndarray]) -> tuple[list[int], dict]:
        """Return (kept_indices, report_dict)."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# --- Tests: Main Problem 1 ---
rng = np.random.default_rng(42)

# Create synthetic images: 3 base images, each with a near-duplicate (small noise)
base_a = rng.integers(50, 200, (64, 64, 3), dtype=np.uint8)
base_b = rng.integers(0, 100, (64, 64, 3), dtype=np.uint8)
base_c = rng.integers(100, 255, (64, 64, 3), dtype=np.uint8)

# Near-duplicates: add small noise (should hash similarly)
dup_a = np.clip(base_a.astype(int) + rng.integers(-2, 3, base_a.shape), 0, 255).astype(np.uint8)
dup_b = np.clip(base_b.astype(int) + rng.integers(-2, 3, base_b.shape), 0, 255).astype(np.uint8)

# Unrelated image
unrelated = rng.integers(0, 255, (64, 64, 3), dtype=np.uint8)

# Images: [base_a(0), dup_a(1), base_b(2), dup_b(3), base_c(4), unrelated(5)]
test_images = [base_a, dup_a, base_b, dup_b, base_c, unrelated]

deduplicator = ImageDeduplicator(threshold=5)

# Test compute_hash: same image -> same hash
h1 = deduplicator.compute_hash(base_a)
h2 = deduplicator.compute_hash(base_a.copy())
assert h1 == h2, "Same image must produce same hash"
assert isinstance(h1, int), "Hash must be an integer"

# Test compute_hash: near-duplicate should have small Hamming distance
h_base = deduplicator.compute_hash(base_a)
h_dup = deduplicator.compute_hash(dup_a)
dist = bin(h_base ^ h_dup).count('1')
assert dist <= 5, f"Near-duplicate Hamming distance too large: {dist}"

# Test find_duplicates
clusters = deduplicator.find_duplicates(test_images)
assert isinstance(clusters, list), "find_duplicates must return a list"
assert all(isinstance(c, set) for c in clusters), "Each cluster must be a set"
assert all(len(c) >= 2 for c in clusters), "Only clusters with 2+ members"

# base_a and dup_a (indices 0,1) should be in the same cluster
found_a_cluster = any({0, 1}.issubset(c) for c in clusters)
assert found_a_cluster, "base_a and dup_a should be in the same cluster"

# Test deduplicate
kept, report = deduplicator.deduplicate(test_images)
assert isinstance(kept, list), "kept_indices must be a list"
assert isinstance(report, dict), "report must be a dict"
assert report['n_input'] == 6, f"n_input should be 6, got {report['n_input']}"
assert report['n_output'] == len(kept), "n_output must match len(kept_indices)"
assert report['n_removed'] == report['n_input'] - report['n_output']
assert report['n_removed'] > 0, "Should have removed at least one duplicate"
assert sorted(kept) == kept, "kept_indices should be sorted"

# No overlap: all kept indices are unique
assert len(set(kept)) == len(kept), "kept_indices has duplicates"

# Kept + removed = all indices
all_indices = set(range(6))
removed_indices = all_indices - set(kept)
assert len(removed_indices) == report['n_removed']

print(f"Kept {len(kept)} of 6 images, removed {report['n_removed']}")
print(f"Clusters: {report['clusters']}")
print("Main Problem 1 PASSED")

---

### Main Problem 2: Re-Captioning Pipeline (System Design as Code)

You're designing a pipeline that re-captions 1 billion video clips using a VLM (vision-language model). Implement the pipeline as a class that can estimate throughput and process shards.

**Class interface:**

```python
class RecaptionPipeline:
    def __init__(self, n_gpus: int, vlm_batch_size: int, clips_per_shard: int): ...
    def estimate_throughput(self) -> dict: ...
    def process_shard(self, shard_clips: list[dict], vlm_fn: Callable) -> list[dict]: ...
    def run_quality_check(self, caption: str) -> float: ...
```

**`estimate_throughput`:**
- Assume each clip takes `vlm_batch_size * 0.5` seconds per batch on one GPU (i.e., 0.5s per clip amortized over a batch is `1 / 0.5 = 2 clips/sec/GPU`)
- Return dict with:
  - `clips_per_sec`: total clips/sec across all GPUs
  - `hours_for_1B`: hours to process 1 billion clips at this rate
  - `gpu_hours`: total GPU-hours (hours_for_1B * n_gpus)

**`process_shard`:**
- Takes a list of clip dicts (each has `clip_id` key and any other metadata)
- Calls `vlm_fn(clip)` for each clip to get a caption string
- Runs `run_quality_check` on each caption
- Returns list of result dicts with: `clip_id`, `new_caption`, `quality_score`, `status` ("ok" if quality_score >= 0.3, else "low_quality")
- Must handle exceptions from `vlm_fn` gracefully: set status="error", new_caption="", quality_score=0.0

**`run_quality_check`:**
- Score a caption on a 0-1 scale based on:
  - Length: captions < 20 chars get penalized, 20-200 is good, > 200 is fine
  - Specificity: bonus if caption contains any of these keywords: color words, numbers, spatial terms ("left", "right", "above", "below", "behind"), or action verbs ("walking", "running", "holding", "moving")
- Return a float in [0, 1]

In [ ]:
class RecaptionPipeline:
    """Re-captioning pipeline for video clips at scale."""

    def __init__(self, n_gpus: int, vlm_batch_size: int, clips_per_shard: int):
        # YOUR CODE HERE
        raise NotImplementedError

    def estimate_throughput(self) -> dict:
        """Estimate processing throughput for 1B clips."""
        # YOUR CODE HERE
        raise NotImplementedError

    def process_shard(self, shard_clips: list[dict], vlm_fn: Callable) -> list[dict]:
        """Process a shard of clips through the VLM and quality check."""
        # YOUR CODE HERE
        raise NotImplementedError

    def run_quality_check(self, caption: str) -> float:
        """Score a caption 0-1 based on length and specificity."""
        # YOUR CODE HERE
        raise NotImplementedError

In [ ]:
# --- Tests: Main Problem 2 ---

pipeline = RecaptionPipeline(n_gpus=8, vlm_batch_size=16, clips_per_shard=1000)

# --- Throughput estimation ---
tp = pipeline.estimate_throughput()
assert isinstance(tp, dict)
assert 'clips_per_sec' in tp
assert 'hours_for_1B' in tp
assert 'gpu_hours' in tp

# With 8 GPUs at 2 clips/sec/GPU = 16 clips/sec total
assert tp['clips_per_sec'] == 16.0, f"Expected 16 clips/sec, got {tp['clips_per_sec']}"

# 1B / 16 clips/sec = 62,500,000 sec = ~17,361 hours
expected_hours = 1_000_000_000 / 16.0 / 3600.0
assert abs(tp['hours_for_1B'] - expected_hours) < 1.0, f"hours_for_1B off: {tp['hours_for_1B']}"

# GPU-hours = hours * n_gpus
assert abs(tp['gpu_hours'] - expected_hours * 8) < 10.0

print(f"Throughput: {tp['clips_per_sec']} clips/sec, {tp['hours_for_1B']:.0f} hours, {tp['gpu_hours']:.0f} GPU-hours")

# --- Quality check ---
# Short generic caption should score low
score_short = pipeline.run_quality_check("a video")
assert 0.0 <= score_short <= 1.0
assert score_short < 0.5, f"Short generic caption scored too high: {score_short}"

# Detailed caption with specifics should score high
detailed = "A person in a red jacket walking along the left side of a cobblestone street, holding a blue umbrella above their head while 3 pigeons fly behind them"
score_detailed = pipeline.run_quality_check(detailed)
assert score_detailed > 0.5, f"Detailed caption scored too low: {score_detailed}"
assert score_detailed > score_short, "Detailed should score higher than short"

print(f"Quality scores: short={score_short:.2f}, detailed={score_detailed:.2f}")

# --- Process shard ---
test_clips = [
    {"clip_id": "clip_001", "path": "/data/clip1.mp4"},
    {"clip_id": "clip_002", "path": "/data/clip2.mp4"},
    {"clip_id": "clip_003", "path": "/data/clip3.mp4"},
    {"clip_id": "clip_004", "path": "/data/clip4.mp4"},
]

def mock_vlm(clip: dict) -> str:
    """Mock VLM that returns captions of varying quality."""
    captions = {
        "clip_001": "A red car driving left on a highway with 5 other vehicles moving behind it",
        "clip_002": "video",  # too short
        "clip_003": "A woman in a blue dress walking right across a park while holding a white dog",
    }
    if clip["clip_id"] == "clip_004":
        raise RuntimeError("GPU OOM")
    return captions[clip["clip_id"]]

results = pipeline.process_shard(test_clips, mock_vlm)
assert len(results) == 4, f"Expected 4 results, got {len(results)}"

# Check result structure
for r in results:
    assert 'clip_id' in r
    assert 'new_caption' in r
    assert 'quality_score' in r
    assert 'status' in r

# clip_001: good caption -> status ok
r1 = next(r for r in results if r['clip_id'] == 'clip_001')
assert r1['status'] == 'ok', f"clip_001 should be ok, got {r1['status']}"

# clip_002: short caption -> low_quality
r2 = next(r for r in results if r['clip_id'] == 'clip_002')
assert r2['status'] == 'low_quality', f"clip_002 should be low_quality, got {r2['status']}"

# clip_004: error -> status error
r4 = next(r for r in results if r['clip_id'] == 'clip_004')
assert r4['status'] == 'error', f"clip_004 should be error, got {r4['status']}"
assert r4['new_caption'] == '', "Error caption should be empty string"
assert r4['quality_score'] == 0.0, "Error quality_score should be 0.0"

print("Shard results:")
for r in results:
    print(f"  {r['clip_id']}: status={r['status']}, score={r['quality_score']:.2f}")
print("Main Problem 2 PASSED")

---

### Main Problem 3: DataLoader Throughput Benchmark (Review: NB00-02)

Implement `measure_loader_throughput(loader, n_batches=20, warmup=3) -> dict`

This is a common utility for benchmarking data pipeline performance — you need to know if your DataLoader is the bottleneck vs. your model.

**Requirements:**
- Iterate through the loader for `warmup + n_batches` batches total
- Skip the first `warmup` batches before starting the timer (they warm up OS caches, worker processes, etc.)
- After warmup, time exactly `n_batches` batches
- Count total samples across the timed batches (use first element of batch, `batch[0].shape[0]`)
- Return dict with:
  - `samples_per_sec`: total_samples / elapsed_sec
  - `total_samples`: int count of samples in timed batches
  - `elapsed_sec`: float wall-clock time for timed batches
- If the loader runs out of batches before `warmup + n_batches`, just use however many timed batches you got
- If no timed batches were collected, return `samples_per_sec=0.0`

In [ ]:
def measure_loader_throughput(
    loader: DataLoader,
    n_batches: int = 20,
    warmup: int = 3,
) -> dict:
    """Benchmark DataLoader throughput, skipping warmup batches before timing."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Tests: Main Problem 3 ---
torch.manual_seed(0)
bench_images = torch.randn(500, 3, 32, 32)
bench_labels = torch.randint(0, 10, (500,))
bench_dataset = TensorDataset(bench_images, bench_labels)
bench_loader = DataLoader(bench_dataset, batch_size=16, shuffle=False)

result = measure_loader_throughput(bench_loader, n_batches=20, warmup=3)

assert isinstance(result, dict)
assert 'samples_per_sec' in result
assert 'total_samples' in result
assert 'elapsed_sec' in result

# Timing should be positive
assert result['elapsed_sec'] > 0, f"Elapsed time should be > 0, got {result['elapsed_sec']}"
assert result['samples_per_sec'] > 0, f"Throughput should be > 0, got {result['samples_per_sec']}"

# We have 500 samples / 16 batch_size = 31 full batches. After 3 warmup, 20 timed = 23 total needed. 31 >= 23.
# 20 timed batches * 16 samples = 320 total samples
assert result['total_samples'] == 320, f"Expected 320 samples, got {result['total_samples']}"

print(f"Throughput: {result['samples_per_sec']:.0f} samples/sec")
print(f"Total samples: {result['total_samples']}, elapsed: {result['elapsed_sec']:.4f}s")

# Test with a small loader that runs out early
small_dataset = TensorDataset(torch.randn(10, 3, 8, 8), torch.zeros(10))
small_loader = DataLoader(small_dataset, batch_size=4)
# 10 / 4 = 2 full batches + 1 partial = 3 batches total. Warmup=3, n_batches=20.
# All 3 batches consumed by warmup -> 0 timed batches.
small_result = measure_loader_throughput(small_loader, n_batches=20, warmup=3)
assert small_result['samples_per_sec'] == 0.0, "Should return 0 when no timed batches"
assert small_result['total_samples'] == 0

print("Main Problem 3 PASSED")

---

## Scoring

| Problem | Points | Criteria |
|---|---|---|
| Warmup 1 | 10 | All assertions pass |
| Warmup 2 | 10 | All assertions pass |
| Warmup 3 | 10 | All assertions pass |
| Main 1 | 25 | All assertions pass, clean union-find or equivalent |
| Main 2 | 25 | All assertions pass, error handling correct |
| Main 3 | 20 | All assertions pass, warmup logic correct |
| **Total** | **100** | |